# 13 · Day 2 종합 캡스톤 — 딥러닝 전처리 파이프라인 (커스텀 커널 + interop)

> **CuPy 2일 집중 코스 — Day 2 / 단원 9 · 마무리 통합 실습 (약 60분)**

Day 2의 핵심을 하나의 **DL 전처리 파이프라인**으로 통합합니다: GPU에서 배치 신호를 표준화→**커스텀 커널** 비선형→
특징 추출(리덕션)→ **무복사로 PyTorch 모델 입력**. 그리고 v0→v1(fuse·메모리)→v2(스트림)로 최적화합니다.

## 무엇을 통합하나
| 단계 | 기법 | 출처 |
|------|------|------|
| 표준화·특징 | ndarray·리덕션 | 02·07 |
| 비선형 커스텀 커널 | `ElementwiseKernel`/`@cupy.fuse` | 07 |
| v1 최적화 | fuse·out=·전송↓ | 05·07 |
| v2 최적화 | 스트림 청크 오버랩 | 06 |
| 모델 입력 | DLPack 무복사 | 12 |

## 목표
- 커스텀 커널을 포함한 전처리를 **end-to-end GPU**로 구성한다.
- v0→v1→v2 단계 최적화 + 정확성 검증 + 무복사 모델 연동.

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare, allclose
print_env()
try:
    import torch; HAS_TORCH = torch.cuda.is_available()
except Exception: HAS_TORCH = False
B, N = 512, 50_000

## Stage 0 — 데이터 생성 (GPU)

(B, N) 잡음 신호 배치. 검증용 host 사본도 둡니다.

In [ ]:
rng = np.random.default_rng(0)
X_np = rng.standard_normal((B, N)).astype(np.float32)
X_cp = cp.asarray(X_np)
print('X:', X_cp.shape)

## Stage 1 — 행별 표준화 (TODO)

각 신호(행)를 z-score 표준화 (02, 브로드캐스팅, 장치 비종속).

In [ ]:
def standardize(X):
    # TODO: xp 선택 후 (X-행평균)/(행std+1e-6)
    raise NotImplementedError

<details><summary>💡 해답 보기</summary>

```python
def standardize(X):
    xp = cp.get_array_module(X)
    mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, keepdims=True)+1e-6
    return (X-mu)/sd
```
</details>

## Stage 2 — 비선형 커스텀 커널 (TODO)

활성화 `g = tanh(z)·exp(-z²)` 를 **커스텀 커널**로(07). 먼저 `ElementwiseKernel`로 작성합니다.

In [ ]:
# TODO: nl = cp.ElementwiseKernel('float32 z','float32 g',
#            'g = tanhf(z) * expf(-z*z);', 'nl_act')
# (또는 @cp.fuse 버전을 v1에서 사용)

<details><summary>💡 해답 보기</summary>

```python
nl = cp.ElementwiseKernel('float32 z','float32 g',
                          'g = tanhf(z) * expf(-z*z);', 'nl_act')
# 검증용 순수 CuPy 참조
def nl_ref(z): return cp.tanh(z)*cp.exp(-z*z)
```
</details>

## Stage 3 — 특징 추출 & v0 조립

활성화 결과의 **행별 평균**을 특징으로 합니다(간단). 전체 파이프라인을 조립하고 CPU/GPU 검증.

In [ ]:
def features_v0(X):
    z = standardize(X)
    g = nl(z) if cp.get_array_module(X) is cp else (np.tanh(z)*np.exp(-z*z))
    return g.mean(axis=1)        # (B,) 특징

# 검증: GPU vs CPU(같은 입력)
# ref = features_v0(X_np); out = cp.asnumpy(features_v0(X_cp))
# allclose(ref, out, rtol=1e-3, atol=1e-3, name='features v0')

## 최적화 v1 — @cupy.fuse + 전송 최소화

비선형을 **융합 커널**로 바꿔 중간배열을 줄이고, 끝까지 GPU에 머무릅니다(05·07).

<details><summary>💡 해답 보기</summary>

```python
@cp.fuse()
def nl_fused(z): return cp.tanh(z)*cp.exp(-z*z)
def features_v1(X):
    z = standardize(X)
    return nl_fused(z).mean(axis=1)
_ = nl_fused(X_cp[:2])   # 워밍업
allclose(cp.asnumpy(features_v0(X_cp)), cp.asnumpy(features_v1(X_cp)), rtol=1e-3, atol=1e-3, name='v1==v0')
```
</details>

## 최적화 v2 — 스트림 청크 오버랩

배치를 청크로 나눠 여러 스트림에서 전처리(06). 청크는 독립이라 겹칠 수 있습니다.

<details><summary>💡 해답 보기</summary>

```python
def features_v2(X, nstreams=3, chunk=128):
    out = cp.empty(X.shape[0], dtype=cp.float32)
    streams = [cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i, s0 in enumerate(range(0, X.shape[0], chunk)):
        with streams[i % nstreams]:
            out[s0:s0+chunk] = nl_fused(standardize(X[s0:s0+chunk])).mean(axis=1)
    for st in streams: st.synchronize()
    return out
allclose(cp.asnumpy(features_v1(X_cp)), cp.asnumpy(features_v2(X_cp)), rtol=1e-3, atol=1e-3, name='v2==v1')
```
</details>

## 모델 입력 — DLPack 무복사 (12)

특징을 **복사 없이** PyTorch로 넘겨 간단한 선형층에 통과시킵니다(torch 있을 때).

In [ ]:
feat = features_v1(X_cp).reshape(B, 1)   # (B,1) 특징
if HAS_TORCH:
    t = torch.from_dlpack(feat)          # zero-copy
    W = torch.randn(1, 8, device='cuda')
    y = t @ W                             # 모델 입력으로 사용
    print('model out:', tuple(y.shape))
else:
    print('torch 없음 — 개념: torch.from_dlpack(feat) 로 무복사 입력')

## 성능 비교 & 도전 과제

v0(CPU)→v0/v1/v2(GPU) 시간을 비교하세요.

<details><summary>도전 과제</summary>

- 특징을 ReductionKernel/cccl `reduce_into`로 바꿔 보기
- 비선형을 RawKernel(11)로 작성해 fuse와 비교
- pinned+blocking=False로 전송까지 오버랩(06)
- 전체를 CUDA Graph로 캡처(06)
</details>

In [ ]:
# (해답 구현 후 실행)
# print_bench(bench(lambda: features_v0(X_np), n_repeat=3, name='v0 CPU'))
# print_bench(bench(lambda: features_v0(X_cp), n_repeat=5, name='v0 GPU'))
# print_bench(bench(lambda: features_v1(X_cp), n_repeat=5, name='v1 fuse'))
# print_bench(bench(lambda: features_v2(X_cp), n_repeat=5, name='v2 streams'))

## 추가 — 특징 커널화 & 단계 프로파일

**실험 — 단계별 이벤트 프로파일**: 표준화/비선형/특징 각 단계 시간을 재 병목을 찾으세요(06).

In [ ]:
import cupy as cp
evs = [cp.cuda.Event() for _ in range(4)]
evs[0].record()
z = standardize(X_cp);            evs[1].record()
g = nl_fused(z);                  evs[2].record()
feat = g.mean(axis=1);            evs[3].record()
evs[3].synchronize()
for nm, a, b in [('standardize',0,1),('nonlin',1,2),('feature',2,3)]:
    print(f'{nm:>11}: {cp.cuda.get_elapsed_time(evs[a], evs[b]):.3f} ms')

**연습 — 특징을 제곱합으로 커널화**: 특징을 `(g*g).mean(axis=1)`(에너지)로 바꾸고, 비선형+제곱을 **하나의 `@cp.fuse`** 로 융합해 v1과 속도를 비교하세요.

In [ ]:
# TODO: @cp.fuse() def nl_sq(z): return (cp.tanh(z)*cp.exp(-z*z))**2
# def features_energy(X): return nl_sq(standardize(X)).mean(axis=1)
# print_bench(bench(lambda: features_energy(X_cp), name='energy(fused)'))

<details><summary>💡 해답 보기</summary>

```python
@cp.fuse()
def nl_sq(z):
    g = cp.tanh(z)*cp.exp(-z*z)
    return g*g
def features_energy(X):
    return nl_sq(standardize(X)).mean(axis=1)
out = features_energy(X_cp)
print_bench(bench(lambda: features_energy(X_cp), n_repeat=5, name='energy(fused)'))
```
</details>

## 제출물 체크리스트

- [ ] Stage 1~2(표준화·커스텀 커널) 구현 + v0 CPU/GPU 일치
- [ ] v1(fuse)·v2(스트림) 구현 + 일치 검증
- [ ] DLPack 무복사로 PyTorch 입력 연결
- [ ] v0(CPU)→v2(GPU) speedup 표
- [ ] (선택) 도전 과제 1개 + 개선 3줄

**2일 코스 완료!** 수고하셨습니다 — NumPy/SciPy 포팅부터 커스텀 CUDA 커널·프레임워크 통합까지 마쳤습니다.